# SRT-M15：手机训练自主底层模型（从零初始化）

本 Notebook **已打包底层实验模型源代码**，无需另行上传 ZIP。使用 Google Colab 执行，模型从随机权重开始。

包含：Transformer、三推理分支、动态通信、内部/外部记忆、可靠标签在线更新、字符级文本生成。第 6 步会安装本地训练问答模块及 DeepSeek 网页/API 可选接入方式。

**这是实验原型，不是通用聊天大模型。外部 DeepSeek 回答不会使 SRT-M15 自动学会知识。网页版桥接需要本地电脑，不能从 Colab 控制手机已登录网页。**


## 第 1 步：解压内嵌的源代码
下面第一段代码将代码展开到当前 Colab 运行环境。

In [ ]:
import io, zipfile, base64, os
SOURCE_ZIP_BASE64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP_BASE64))) as z:
    z.extractall("/content")
os.chdir("/content/SRT_M15_Starter")
print("工作目录:", os.getcwd())
print("发现的源码:", sorted(os.listdir("srt_m15")))


## 第 2 步：测试网络结构
单元测试验证梯度、因果注意力不泄露未来信息、独立网络可关闭通信，以及在线更新等。

In [ ]:
!python -m unittest discover -s tests -v


## 第 3 步：从零训练三个协作神经网络
小任务：网络 A 看到第一个数字，B 看到第二个数字，C 看不到两个数字；**只有 C 的答案用来训练**。如果 C 学会加法，就必须通过 A/B 的内部通信获得信息。

In [ ]:
!python train.py --steps 180 --task distributed --device auto --log-every 45


## 第 4 步：测试推理、路由、记忆、在线更新与消融

In [ ]:
!python demo.py


## 第 5 步（可选）：从零训练字符级语言生成
这是另一条模型训练路径，只学习小型示范语料中的字符规律。它不会成为 ChatGPT 级别的模型。

In [ ]:
!python train_lm.py --steps 65 --device auto --log-every 20


In [ ]:
!python generate.py --prompt "你好" --new-tokens 25


## 第 6 步：安装本地模型、多来源聊天和 HTTP API

此单元把训练模块、DeepSeek 可选桥接和新增的 `api_server.py` 安装到当前 Colab 环境，不会将密码上传 GitHub。SRT-M15 本地模型和外部模型使用不同的模型 ID，不能将 DeepSeek 的回答当成 SRT-M15 训练结果。


In [ ]:
# 第6步：安装本地训练、多来源聊天和 DeepSeek 可选接入模块
from pathlib import Path
Path("srt_m15").mkdir(exist_ok=True)
Path("srt_m15/chat.py").write_text("\"\"\"SRT-M15 local neural chat with explicit, opt-in external LLM backends.\n\nLocal answers come from the experimental SRT-M15 weights, not fixed identity\nstrings. DeepSeek Web/API replies are labeled as external and never silently\ntreated as learned SRT-M15 knowledge.\n\"\"\"\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport os\nimport re\nimport unicodedata\n\nimport torch\nfrom .model import ModelConfig, SRTM15\nfrom .tasks import ArithmeticTokenizer\nfrom .language import CharVocabulary, SRTM15LanguageModel\n\n\n@dataclass\nclass ChatReply:\n    mode: str\n    text: str\n    provider: str = \"SRT-M15\"\n\n\ndef recognize_question(message):\n    cleaned = unicodedata.normalize(\"NFKC\", message).strip()\n    match = re.search(r\"(?<![\\d-])(\\d+)\\s*(?:\\+|加)\\s*(\\d+)(?!\\d)\", cleaned)\n    if match:\n        rest = cleaned[match.end():]\n        if re.search(r\"(?:[+*/\\-]|加|减|乘|除)\\s*\\d+\", rest):\n            return (\"unsupported_expression\",)\n        a, b = int(match.group(1)), int(match.group(2))\n        if a <= 9 and b <= 9:\n            return (\"addition\", a, b)\n        return (\"unsupported_math\", a, b)\n    if re.search(r\"\\d+\\s*(?:[-*/]|减|乘|除)\\s*\\d+\", cleaned):\n        return (\"unsupported_expression\",)\n    return (\"text\",)\n\n\ndef _is_identity_question(message):\n    \"\"\"Route identity to local weights in auto mode (not a scripted answer).\"\"\"\n    normalized = unicodedata.normalize(\"NFKC\", message).casefold()\n    normalized = re.sub(r\"[\\W_]+\", \"\", normalized)\n    return normalized in {\n        \"你是谁\", \"你是谁呀\", \"你是谁啊\", \"你到底是谁\", \"你好你是谁\",\n        \"你叫什么\", \"你叫什么名字\", \"你叫什么名\", \"你的名字\",\n        \"你的名字是什么\", \"你的身份是什么\", \"你的身份\",\n        \"你是什么\", \"你是什么模型\", \"你是哪个模型\", \"你是什么ai\",\n        \"请介绍一下你自己\", \"介绍一下你自己\", \"自我介绍\",\n        \"自我介绍一下\", \"你是gpt吗\", \"你是chatgpt吗\",\n        \"whoareyou\", \"whatisyourname\", \"introduceyourself\",\n    }\n\n\nclass SRTChatEngine:\n    VALID_BACKENDS = {\"local\", \"web\", \"api\", \"auto\"}\n\n    def __init__(self, root=None, arithmetic_path=None, language_path=None,\n                 backend=\"local\", external_backend=None):\n        self.root = Path(root or \".\").resolve()\n        self.arithmetic_path = (Path(arithmetic_path) if arithmetic_path\n                                else self.root / \"checkpoints/srt_m15_toy.pt\")\n        if language_path is None:\n            enhanced = self.root / \"checkpoints/srt_m15_qa.pt\"\n            language_path = enhanced if enhanced.is_file() else self.root / \"checkpoints/srt_m15_lm.pt\"\n        self.language_path = Path(language_path)\n        self.backend = str(backend).strip().lower()\n        self.external_backend = str(\n            external_backend or os.environ.get(\"SRT_EXTERNAL_BACKEND\", \"api\")\n        ).strip().lower()\n        if self.backend not in self.VALID_BACKENDS:\n            raise ValueError(\"backend 必须是 local / web / api / auto\")\n        if self.external_backend not in {\"web\", \"api\"}:\n            raise ValueError(\"external_backend 只能是 web 或 api\")\n        self._math = None\n        self._lm = None\n        self._vocab = None\n        self._web_client = None\n        torch.set_num_threads(max(1, min(torch.get_num_threads(), 4)))\n\n    def status(self):\n        return {\n            \"model\": \"SRT-M15\",\n            \"backend\": self.backend,\n            \"external_backend\": self.external_backend,\n            \"math_checkpoint\": self.arithmetic_path.is_file(),\n            \"language_checkpoint\": self.language_path.is_file(),\n            \"capabilities\": [\"0..9 addition\", \"trainable char-level QA\",\n                             \"optional DeepSeek Web (desktop)\", \"optional DeepSeek API\"],\n        }\n\n    def _math_model(self):\n        if self._math is None:\n            if not self.arithmetic_path.is_file():\n                return None\n            data = torch.load(self.arithmetic_path, map_location=\"cpu\", weights_only=True)\n            model = SRTM15(ModelConfig(**data[\"config\"]))\n            model.load_state_dict(data[\"model\"])\n            self._math = (model.eval(), data.get(\"task\") == \"distributed\")\n        return self._math\n\n    def _language_model(self):\n        if self._lm is None:\n            if not self.language_path.is_file():\n                return None\n            data = torch.load(self.language_path, map_location=\"cpu\", weights_only=True)\n            vocab = CharVocabulary(data[\"vocab\"])\n            model = SRTM15LanguageModel(ModelConfig(**data[\"config\"]))\n            model.load_state_dict(data[\"model\"])\n            self._lm = model.eval()\n            self._vocab = vocab\n        return self._lm\n\n    def _external_reply(self, question, backend):\n        if backend == \"web\":\n            from .deepseek_web import DeepSeekWebClient, DeepSeekWebError\n            if self._web_client is None:\n                self._web_client = DeepSeekWebClient()\n            try:\n                answer = self._web_client.ask(question)\n                return ChatReply(\"external\", answer, \"DeepSeek Web\")\n            except DeepSeekWebError as exc:\n                return ChatReply(\"error\", f\"网页版 DeepSeek 未成功回答：{exc}\", \"DeepSeek Web\")\n        if backend == \"api\":\n            from .deepseek_api import DeepSeekAPIClient, DeepSeekAPIError\n            try:\n                answer = DeepSeekAPIClient().ask(question)\n                return ChatReply(\"external\", answer, \"DeepSeek API\")\n            except DeepSeekAPIError as exc:\n                return ChatReply(\"error\", f\"DeepSeek API 未成功回答：{exc}\", \"DeepSeek API\")\n        raise ValueError(\"未知外部模型方式\")\n\n    def close(self):\n        if self._web_client is not None:\n            self._web_client.close()\n            self._web_client = None\n\n    def reply(self, message, *, new_tokens=80, backend=None):\n        message = str(message).strip()\n        if len(message) > 400:\n            return ChatReply(\"limit\", \"输入过长，请将问题缩短到400个字符以内。\")\n        if not message:\n            return ChatReply(\"notice\", \"请输入问题，例如：你是谁，或者：1+9=多少。\")\n\n        # /raw explicitly tests SRT-M15 weights even if web/api/auto is selected.\n        raw_mode = message.casefold().startswith(\"/raw \")\n        if raw_mode:\n            message = message[5:].strip()\n            if not message:\n                return ChatReply(\"notice\", \"请在 /raw 后输入问题，例如 /raw 你是谁。\")\n\n        selected = str(backend or self.backend).strip().lower()\n        if selected not in self.VALID_BACKENDS:\n            raise ValueError(\"backend 必须是 local / web / api / auto\")\n        if raw_mode:\n            selected = \"local\"\n\n        route = recognize_question(message)\n        if selected == \"auto\":\n            if route[0] == \"addition\" or _is_identity_question(message):\n                selected = \"local\"\n            else:\n                selected = self.external_backend\n        if selected in {\"web\", \"api\"}:\n            return self._external_reply(message, selected)\n\n        if route[0] == \"unsupported_expression\":\n            return ChatReply(\"limit\", \"当前本地数学模型仅支持两个 0～9 整数的单次加法；可切换网页版或 API。\")\n        if route[0] == \"unsupported_math\":\n            a, b = route[1:]\n            return ChatReply(\"limit\", f\"识别为 {a}+{b}，但本地数学网络仅训练了 0～9 的两个整数相加。\")\n        if route[0] == \"addition\":\n            loaded = self._math_model()\n            if loaded is None:\n                return ChatReply(\"notice\", \"没有数学模型权重，请先训练或恢复 SRT_M15_Results.zip。\")\n            model, distributed = loaded\n            a, b = route[1:]\n            tokens, _ = ArithmeticTokenizer.batch([(a, b)], distributed=distributed)\n            with torch.inference_mode():\n                result = model(tokens, hard_routes=True)\n            logits = result[\"receiver_logits\"] if distributed else result[\"logits\"]\n            predicted = int(logits.argmax(-1).item())\n            verified = ArithmeticTokenizer.verify(a, b)\n            verdict = \"预测正确\" if predicted == verified else \"预测错误（研究实验模型）\"\n            return ChatReply(\"math\", f\"题目：{a}+{b}\\n神经网络预测：{predicted}\\n验证器答案：{verified}\\n{verdict}\")\n\n        model = self._language_model()\n        if model is None:\n            return ChatReply(\n                \"notice\",\n                \"没有本地语言模型权重。请在 Notebook 第 9 步运行 train_qa.py，\"\n                \"或恢复之前训练好的模型。\"\n            )\n\n        unknown = sorted({ch for ch in message if ch not in self._vocab.stoi})\n        notice = \"\"\n        if unknown:\n            sample = \" \".join(repr(ch) for ch in unknown[:8])\n            if \"□\" not in self._vocab.stoi:\n                return ChatReply(\"vocab\", f\"模型词表缺少：{sample}。请训练第 9 步问答模型。\")\n            message = \"\".join(ch if ch in self._vocab.stoi else \"□\" for ch in message)\n            notice = f\"未知字符 {sample} 被替换为 □，可能影响答案。\\n\"\n\n        framed = f\"问：{message}\\n答：\"\n        prompt = framed if all(ch in self._vocab.stoi for ch in \"问：\\n答\") else message\n        prompt = prompt[-model.cfg.max_length:]\n        input_ids = torch.tensor([self._vocab.encode(prompt)], dtype=torch.long)\n\n        # Same greedy strategy as train_qa.py; no hard-coded identity string.\n        output_chars = []\n        with torch.inference_mode():\n            for _ in range(max(1, min(int(new_tokens), 80))):\n                logits = model(input_ids[:, -model.cfg.max_length:])[\"logits\"][:, -1]\n                next_id = int(logits.argmax(-1).item())\n                character = self._vocab.chars[next_id]\n                if character == \"\\n\":\n                    break\n                output_chars.append(character)\n                input_ids = torch.cat(\n                    [input_ids, torch.tensor([[next_id]], dtype=torch.long)], dim=1\n                )\n        answer = \"\".join(output_chars)\n        if not answer:\n            answer = \"（神经网络没有生成有效回答，请继续训练。）\"\n        return ChatReply(\"text\", notice + answer)\n", encoding="utf-8")
Path("srt_m15/deepseek_web.py").write_text("\"\"\"Opt-in DeepSeek WEB assistant via a visible local browser.\n\nThis is NOT the official API. No private endpoints, auth-token extraction,\ncookie transfer, CAPTCHA bypass, or headless stealth are used.\nIt requires Playwright and a local desktop with a user-interactive browser.\nIt cannot control a separately logged-in phone browser from Colab.\n\"\"\"\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport os\nimport sys\nimport time\n\n\nclass DeepSeekWebError(RuntimeError):\n    \"\"\"The DeepSeek web page was unavailable or could not be read.\"\"\"\n\n\nclass DeepSeekWebClient:\n    URL = \"https://chat.deepseek.com/\"\n\n    def __init__(self, *, profile_dir=None, timeout=120):\n        self.profile_dir = Path(\n            profile_dir or os.environ.get(\"SRT_DEEPSEEK_WEB_PROFILE\")\n            or Path.home() / \".srt_m15\" / \"deepseek_web\"\n        ).expanduser()\n        self.timeout = max(10, int(timeout))\n        self._executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix=\"deepseek-web\")\n        self._pw = None\n        self._browser = None\n        self._page = None\n\n    def _check_environment(self):\n        if os.environ.get(\"COLAB_RELEASE_TAG\") or os.environ.get(\"COLAB_GPU\"):\n            raise DeepSeekWebError(\n                \"Google Colab 运行在云端，无法直接使用你手机上已登录的 DeepSeek 网页。\"\n                \"请在本地电脑执行网页版桥接，或者在 Colab 使用官方 API。\"\n            )\n        if sys.platform.startswith(\"linux\") and not (\n            os.environ.get(\"DISPLAY\") or os.environ.get(\"WAYLAND_DISPLAY\")\n        ):\n            raise DeepSeekWebError(\"网页版桥接需要本地电脑的可视化桌面环境。\")\n\n    def _ensure_browser(self):\n        self._check_environment()\n        if self._page is not None and not self._page.is_closed():\n            return self._page\n        try:\n            from playwright.sync_api import sync_playwright\n        except ImportError as exc:\n            raise DeepSeekWebError(\n                \"请先安装: pip install playwright && python -m playwright install chromium\"\n            ) from exc\n        self.profile_dir.mkdir(mode=0o700, parents=True, exist_ok=True)\n        try:\n            os.chmod(self.profile_dir, 0o700)\n        except OSError:\n            pass\n        try:\n            self._pw = sync_playwright().start()\n            self._browser = self._pw.chromium.launch_persistent_context(\n                user_data_dir=str(self.profile_dir), headless=False,\n                viewport={\"width\": 1200, \"height\": 850},\n            )\n            self._page = self._browser.pages[0] if self._browser.pages else self._browser.new_page()\n            self._page.goto(self.URL, wait_until=\"domcontentloaded\", timeout=45000)\n            return self._page\n        except Exception as exc:\n            raise DeepSeekWebError(\n                \"无法启动 DeepSeek 本地浏览器。请确认已经安装 Chromium，\"\n                \"未在另一个进程中占用该浏览器配置目录。\"\n            ) from exc\n\n    def _find_composer(self, page):\n        # These are public DOM controls, not DeepSeek's undocumented network API.\n        for selector in ('textarea', '[contenteditable=\"true\"]', '[role=\"textbox\"]'):\n            candidates = page.locator(selector)\n            for idx in range(candidates.count() - 1, -1, -1):\n                candidate = candidates.nth(idx)\n                try:\n                    if candidate.is_visible() and candidate.is_enabled():\n                        return candidate\n                except Exception:\n                    continue\n        return None\n\n    def _login(self):\n        page = self._ensure_browser()\n        print(\"请在打开的 DeepSeek 官方网页手动登录。不要把密码提供给 SRT-M15。\")\n        input(\"网页登录完成、出现聊天输入框后，回到终端按 Enter：\")\n        page.reload(wait_until=\"domcontentloaded\", timeout=45000)\n        if self._find_composer(page) is None:\n            raise DeepSeekWebError(\"尚未检测到聊天输入框，请检查是否已经登录。\")\n        return \"网页版已准备就绪；浏览器登录资料仅保留在本机独立配置目录。\"\n\n    def login(self):\n        \"\"\"Manually log in from the visible browser. Call once on a desktop.\"\"\"\n        return self._executor.submit(self._login).result()\n\n    @staticmethod\n    def _answer_blocks(page):\n        # Current pages may rename classes: fail clearly instead of scraping\n        # unrelated elements or silently returning the user's own question.\n        for selector in (\n            \".ds-markdown\",\n            '[data-role=\"assistant\"] .markdown',\n            '[data-message-role=\"assistant\"]',\n        ):\n            loc = page.locator(selector)\n            if loc.count():\n                return loc\n        return page.locator(\".ds-markdown\")\n\n    def _ask(self, question):\n        page = self._ensure_browser()\n        if \"sign_in\" in page.url or \"login\" in page.url:\n            raise DeepSeekWebError(\n                \"请先在本地电脑运行 python deepseek_web_cli.py --login，\"\n                \"自行登录网页版 DeepSeek。\"\n            )\n        composer = self._find_composer(page)\n        if composer is None:\n            raise DeepSeekWebError(\n                \"未找到 DeepSeek 网页输入框。请确认已登录；\"\n                \"网页布局可能发生变化，需要更新选择器。\"\n            )\n        blocks = self._answer_blocks(page)\n        previous_count = blocks.count()\n        composer.fill(question)\n        composer.press(\"Enter\")\n        started = time.monotonic()\n        previous_text = \"\"\n        stable_since = None\n        while time.monotonic() - started < self.timeout:\n            page.wait_for_timeout(700)\n            blocks = self._answer_blocks(page)\n            if blocks.count() <= previous_count:\n                continue\n            answer = blocks.last.inner_text(timeout=3000).strip()\n            if not answer:\n                continue\n            if answer == previous_text:\n                if stable_since and time.monotonic() - stable_since >= 3.0:\n                    return answer\n            else:\n                previous_text = answer\n                stable_since = time.monotonic()\n        raise DeepSeekWebError(\n            \"等待网页版回答超时，或页面 DOM 结构已变化。\"\n            \"请直接查看浏览器，不会擅自重发同一问题。\"\n        )\n\n    def ask(self, question):\n        if not isinstance(question, str) or not question.strip():\n            raise ValueError(\"问题不能为空\")\n        return self._executor.submit(self._ask, question.strip()).result(\n            timeout=self.timeout + 65\n        )\n\n    def _close(self):\n        try:\n            if self._browser is not None:\n                self._browser.close()\n        finally:\n            self._browser = None\n            self._page = None\n            if self._pw is not None:\n                self._pw.stop()\n                self._pw = None\n\n    def close(self):\n        try:\n            self._executor.submit(self._close).result(timeout=15)\n        finally:\n            self._executor.shutdown(wait=False)\n", encoding="utf-8")
Path("srt_m15/deepseek_api.py").write_text("\"\"\"Optional official DeepSeek API adapter. The browser bridge does not need this.\"\"\"\nimport json\nimport os\nfrom urllib import error, request\n\n\nclass DeepSeekAPIError(RuntimeError):\n    pass\n\n\nclass DeepSeekAPIClient:\n    URL = \"https://api.deepseek.com/chat/completions\"\n\n    def __init__(self, api_key=None, model=None, timeout=70):\n        self.api_key = api_key or os.getenv(\"DEEPSEEK_API_KEY\", \"\")\n        self.model = model or os.getenv(\"DEEPSEEK_MODEL\", \"deepseek-flash\")\n        self.timeout = max(5, int(timeout))\n\n    def ask(self, prompt):\n        if not self.api_key:\n            raise DeepSeekAPIError(\n                \"没有 DEEPSEEK_API_KEY。网页版不需要 API Key，\"\n                \"但本地桌面网页版模式需要安装 Playwright 并手动登录。\"\n            )\n        body = json.dumps({\n            \"model\": self.model,\n            \"messages\": [{\"role\": \"user\", \"content\": str(prompt)}],\n            \"stream\": False,\n        }).encode(\"utf-8\")\n        req = request.Request(\n            self.URL, data=body, method=\"POST\",\n            headers={\n                \"Content-Type\": \"application/json\",\n                \"Authorization\": f\"Bearer {self.api_key}\",\n            }\n        )\n        try:\n            with request.urlopen(req, timeout=self.timeout) as response:\n                payload = json.load(response)\n        except error.HTTPError as exc:\n            raise DeepSeekAPIError(f\"DeepSeek API 返回 HTTP {exc.code}。\") from exc\n        except (error.URLError, TimeoutError) as exc:\n            raise DeepSeekAPIError(f\"DeepSeek API 网络连接失败：{type(exc).__name__}\") from exc\n        try:\n            content = payload[\"choices\"][0][\"message\"][\"content\"]\n            if not isinstance(content, str) or not content.strip():\n                raise ValueError(\"empty answer\")\n            return content.strip()\n        except (KeyError, IndexError, TypeError, ValueError) as exc:\n            raise DeepSeekAPIError(\"DeepSeek API 未返回有效文本。\") from exc\n", encoding="utf-8")
Path("chat_ui.py").write_text("\"\"\"SRT-M15 notebook chat UI: local, DeepSeek Web, or official API.\n\nWeb mode requires a visible browser on a local desktop, not remote Colab.\nExternal answers are prominently labeled; they do not train the local model.\n\"\"\"\nfrom html import escape\nfrom IPython.display import display\nimport ipywidgets as widgets\nfrom srt_m15.chat import SRTChatEngine\n\ntry:\n    from google.colab import output\n    output.enable_custom_widget_manager()\nexcept ImportError:\n    pass\n\nengine = SRTChatEngine()\nmessages = []\n\nhistory = widgets.HTML(\n    value=\"<div style='padding:12px'>试试：<b>你是谁</b>、<b>1+9=多少</b>。\"\n          \"本地答案来自实验模型；DeepSeek 答案将注明外部来源。</div>\",\n    layout=widgets.Layout(width=\"100%\")\n)\ntextbox = widgets.Textarea(\n    value=\"\", placeholder=\"输入问题，例如：你是谁\",\n    description=\"提问：\", layout=widgets.Layout(width=\"100%\", height=\"100px\")\n)\nbackend_select = widgets.Dropdown(\n    options=[\n        (\"SRT-M15 本地神经网络\", \"local\"),\n        (\"DeepSeek 网页（仅本机电脑）\", \"web\"),\n        (\"DeepSeek API（可在 Colab 使用）\", \"api\"),\n        (\"自动路由（本地/外部）\", \"auto\"),\n    ],\n    value=\"local\", description=\"回答方式：\",\n    layout=widgets.Layout(width=\"100%\")\n)\nsend = widgets.Button(description=\"发送问题\", button_style=\"primary\", icon=\"paper-plane\")\nclear = widgets.Button(description=\"清空对话\", icon=\"trash\")\nstatus = widgets.HTML(\n    value=\"<small>默认只使用 SRT-M15。切换到外部模式才会将问题发送给 DeepSeek。\"\n          \"网页版需要本地电脑手动登录，不支持从 Colab 接管手机浏览器。</small>\"\n)\n\n\ndef render_history():\n    bubbles = []\n    for who, msg in messages[-30:]:\n        is_user = who == \"你\"\n        bg = \"#e5f2ff\" if is_user else \"#f0f3f5\"\n        safe = escape(msg).replace(\"\\n\", \"<br>\")\n        bubbles.append(\n            f'<div style=\"margin:8px 4px;padding:12px;background:{bg};'\n            f'border-radius:12px\"><b>{escape(who)}</b><br>{safe}</div>'\n        )\n    history.value = (\n        '<div style=\"max-height:400px;overflow-y:auto\">'\n        + \"\".join(bubbles) + \"</div>\"\n    )\n\n\ndef on_send(_):\n    question = textbox.value.strip()\n    if not question:\n        status.value = \"<small>请先输入问题。</small>\"\n        return\n    textbox.value = \"\"\n    send.disabled = True\n    try:\n        result = engine.reply(question, backend=backend_select.value)\n        reply = result.text\n        label = result.provider + (\"（错误）\" if result.mode == \"error\" else \"\")\n        status.value = (\n            f\"<small>本次回答来源：<b>{escape(result.provider)}</b>。\"\n            \"外部回答不是 SRT-M15 独立生成，也不会自动训练它。</small>\"\n        )\n    except Exception as exc:\n        reply = f\"模型加载或推理错误：{type(exc).__name__}: {exc}\"\n        label = \"系统提示\"\n    finally:\n        send.disabled = False\n    messages.extend([(\"你\", question), (label, reply)])\n    render_history()\n\n\ndef on_clear(_):\n    messages.clear()\n    render_history()\n\n\nsend.on_click(on_send)\nclear.on_click(on_clear)\ndisplay(widgets.VBox([\n    widgets.HTML(\"<h3>SRT-M15 多来源问答实验</h3>\"),\n    history, backend_select, textbox, widgets.HBox([send, clear]), status\n], layout=widgets.Layout(width=\"100%\")))\n", encoding="utf-8")
Path("train_qa.py").write_text("\"\"\"Supervised character-level QA training for the SRT-M15 research model.\n\nNo external LLM, pretrained weights, or hard-coded identity reply at inference.\nQuestion/prompt characters are context; only answer tokens contribute to loss.\nThe tiny model can still fail, especially on paraphrases not seen in training.\n\"\"\"\nimport argparse\nimport random\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\n\nfrom srt_m15 import ModelConfig\nfrom srt_m15.language import CharVocabulary, SRTM15LanguageModel\n\n\nIDENTITY_ANSWER = \"我是SRT-M15，一个从零训练的小型Transformer实验模型。\"\nIDENTITY_TRAIN = [\n    \"你是谁\", \"你是谁呀\", \"你叫什么\", \"你叫什么名字\",\n    \"你的名字是什么\", \"你的身份是什么\", \"你是什么模型\",\n    \"你是哪个模型\", \"你是什么AI\", \"请介绍一下你自己\",\n    \"介绍一下你自己\", \"自我介绍\", \"你是什么\",\n]\nOTHER_TRAIN = [\n    (\"你好\", \"你好！\"),\n    (\"早上好\", \"早上好！\"),\n    (\"你会什么\", \"我可以尝试字符级问答和简单加法。\"),\n    (\"你会算数学吗\", \"我正在学习简单计算。\"),\n    (\"什么是模型\", \"模型是学习数据规律的计算系统。\"),\n    (\"什么是训练\", \"训练通过调整参数来减小误差。\"),\n    (\"什么是推理\", \"推理是利用模型参数预测结果。\"),\n    (\"什么是人工智能\", \"人工智能研究如何让机器完成智能任务。\"),\n    (\"什么是记忆\", \"记忆用于保存和使用信息。\"),\n    (\"你能聊天吗\", \"我能尝试简单问答，但能力有限。\"),\n    (\"你是ChatGPT吗\", \"不是，我是SRT-M15实验模型。\"),\n    (\"你是GPT吗\", \"不是，我是SRT-M15实验模型。\"),\n    (\"为什么会回答错\", \"可能因为训练不足或输入超出能力范围。\"),\n]\n# Development examples are excluded from training. Sharing the identity *answer*\n# tests unseen question phrasing, not ability to discover facts on its own.\nDEV = [\n    (\"你到底是谁\", IDENTITY_ANSWER),\n    (\"能介绍你自己吗\", IDENTITY_ANSWER),\n    (\"你自己的名字是什么\", IDENTITY_ANSWER),\n    (\"你好呀\", \"你好！\"),\n    (\"你能做加法吗\", \"我正在学习简单计算。\"),\n]\n\n\ndef training_pairs():\n    return [(q, IDENTITY_ANSWER) for q in IDENTITY_TRAIN] + OTHER_TRAIN\n\n\ndef make_vocab():\n    # Include development characters for evaluation without unknown-token errors.\n    text = \"\".join(\"问：\" + q + \"\\n答：\" + a + \"\\n\" for q, a in training_pairs() + DEV)\n    return CharVocabulary.from_text(text + \"□\")\n\n\ndef make_example(vocab, question, answer, context):\n    prompt = f\"问：{question}\\n答：\"\n    whole = prompt + answer + \"\\n\"\n    if len(whole) - 1 > context:\n        raise ValueError(\n            f\"问答长度 {len(whole) - 1} 超过 --context {context}，\"\n            \"请增大 context 或缩短示例。\"\n        )\n    indices = vocab.encode(whole)\n    inputs = torch.tensor(indices[:-1], dtype=torch.long)\n    labels = torch.tensor(indices[1:], dtype=torch.long)\n    # Label position i predicts character i+1; first answer label at len(prompt)-1.\n    labels[:len(prompt) - 1] = -100\n    return inputs, labels\n\n\ndef collate(examples, pad_id, device):\n    length = max(x.numel() for x, _ in examples)\n    x = torch.full((len(examples), length), pad_id, dtype=torch.long)\n    y = torch.full((len(examples), length), -100, dtype=torch.long)\n    for idx, (tokens, targets) in enumerate(examples):\n        x[idx, :len(tokens)] = tokens\n        y[idx, :len(targets)] = targets\n    return x.to(device), y.to(device)\n\n\n@torch.inference_mode()\ndef answer(model, vocab, question, device, max_new_tokens=80):\n    \"\"\"Greedy model decoding until newline: never substitute a canned answer.\"\"\"\n    model.eval()\n    prompt = f\"问：{question}\\n答：\"\n    unknown = [c for c in prompt if c not in vocab.stoi]\n    if unknown:\n        return f\"[词表缺少: {unknown[0]!r}]\"\n    seq = torch.tensor([vocab.encode(prompt)], dtype=torch.long, device=device)\n    response = []\n    for _ in range(max_new_tokens):\n        logits = model(seq[:, -model.cfg.max_length:])[\"logits\"][:, -1, :]\n        next_id = int(logits.argmax(-1).item())\n        character = vocab.chars[next_id]\n        if character == \"\\n\":\n            break\n        response.append(character)\n        next_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)\n        seq = torch.cat((seq, next_tensor), dim=1)\n    return \"\".join(response)\n\n\n@torch.inference_mode()\ndef evaluate(model, vocab, dev_examples, pad_id, device, context):\n    model.eval()\n    total_loss, count = 0.0, 0\n    for question, reply in DEV:\n        x, y = collate([make_example(vocab, question, reply, context)],\n                       pad_id, device)\n        logits = model(x)[\"logits\"].reshape(-1, len(vocab.chars))\n        total_loss += F.cross_entropy(logits, y.reshape(-1),\n                                      ignore_index=-100, reduction=\"sum\").item()\n        count += int((y != -100).sum().item())\n    cases = [\n        (\"你是谁\", IDENTITY_ANSWER),\n        (\"你叫什么名字\", IDENTITY_ANSWER),\n        DEV[0],\n        DEV[1],\n        (\"你是ChatGPT吗\", \"不是，我是SRT-M15实验模型。\"),\n    ]\n    outputs = [(q, answer(model, vocab, q, device), expected)\n               for q, expected in cases]\n    identity_correct = sum(actual == expected for _, actual, expected in outputs[:4])\n    dev_correct = sum(actual == expected for _, actual, expected in outputs[2:4])\n    return {\n        \"dev_answer_loss\": round(total_loss / max(count, 1), 5),\n        \"identity_exact\": identity_correct,\n        \"identity_total\": 4,\n        \"unseen_identity_exact\": dev_correct,\n        \"unseen_identity_total\": 2,\n        \"examples\": outputs,\n    }\n\n\ndef save(model, vocab, cfg, path, step, evaluation):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    state = {name: tensor.detach().cpu().clone()\n             for name, tensor in model.state_dict().items()}\n    torch.save({\n        \"model\": state,\n        \"config\": asdict(cfg),\n        \"vocab\": vocab.chars,\n        \"qa_training\": \"masked_answer_v1\",\n        \"step\": step,\n        \"evaluation\": evaluation,\n    }, path)\n\n\ndef load(path, device):\n    data = torch.load(path, map_location=\"cpu\", weights_only=True)\n    vocab = CharVocabulary(data[\"vocab\"])\n    cfg = ModelConfig(**data[\"config\"])\n    model = SRTM15LanguageModel(cfg)\n    model.load_state_dict(data[\"model\"])\n    return model.to(device).eval(), vocab, cfg, data\n\n\ndef run(args):\n    if args.steps <= 0 or args.batch_size <= 0 or args.eval_every <= 0 or args.log_every <= 0:\n        raise ValueError(\"训练步数、batch size 和日志间隔必须大于 0\")\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.set_num_threads(max(1, min(torch.get_num_threads(), 4)))\n    device = (\"cuda\" if torch.cuda.is_available() else \"cpu\") if args.device == \"auto\" else args.device\n    output = Path(args.output)\n\n    if args.eval_only:\n        model, vocab, cfg, data = load(output, device)\n        print(\"已加载检查点：\", output, \"step=\", data.get(\"step\"),\n              \"训练方式=\", data.get(\"qa_training\", \"旧式字符续写\"))\n        if data.get(\"qa_training\") != \"masked_answer_v1\":\n            print(\"警告：这不是新的答案监督训练检查点。\")\n        for question in [\"你是谁\", \"你叫什么名字\", \"你到底是谁\", \"你是ChatGPT吗\", \"你好\"]:\n            print(f\"问：{question}\\n模型答：{answer(model, vocab, question, device)}\")\n        return\n\n    vocab = make_vocab()\n    cfg = ModelConfig(\n        vocab_size=len(vocab.chars),\n        num_classes=len(vocab.chars),\n        dim=args.dim,\n        max_length=args.context,\n        reasoning_steps=args.reasoning_steps,\n    )\n    model = SRTM15LanguageModel(cfg).to(device)\n    pad_id = vocab.stoi[\"□\"]\n    train_examples = [make_example(vocab, q, a, args.context)\n                      for q, a in training_pairs()]\n    dev_examples = [make_example(vocab, q, a, args.context) for q, a in DEV]\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)\n    print(f\"device={device} vocab={len(vocab.chars)} \"\n          f\"params={sum(p.numel() for p in model.parameters()):,} \"\n          f\"train_qa={len(train_examples)} dev_qa={len(dev_examples)}\")\n    best_score = None\n\n    for step in range(1, args.steps + 1):\n        model.train()\n        batch = random.choices(train_examples, k=args.batch_size)\n        x, y = collate(batch, pad_id, device)\n        logits = model(x)[\"logits\"]\n        loss = F.cross_entropy(logits.reshape(-1, len(vocab.chars)),\n                               y.reshape(-1), ignore_index=-100)\n        optimizer.zero_grad(set_to_none=True)\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        optimizer.step()\n        if step % args.log_every == 0 or step == 1:\n            print(f\"step={step}/{args.steps} train_answer_loss={loss.item():.4f}\", flush=True)\n        if step % args.eval_every == 0 or step == args.steps:\n            metrics = evaluate(model, vocab, dev_examples, pad_id, device, args.context)\n            # Prioritize correct identity generation; break ties using dev answer loss.\n            score = (metrics[\"identity_exact\"], metrics[\"unseen_identity_exact\"],\n                     -metrics[\"dev_answer_loss\"])\n            print(f\"eval step={step} identity={metrics['identity_exact']}/4 \"\n                  f\"unseen={metrics['unseen_identity_exact']}/2 \"\n                  f\"dev_loss={metrics['dev_answer_loss']:.4f}\", flush=True)\n            for q, actual, expected in metrics[\"examples\"][:3]:\n                print(f\"  {q} -> {actual!r} ({'✓' if actual == expected else '未匹配'})\", flush=True)\n            if best_score is None or score > best_score:\n                best_score = score\n                save(model, vocab, cfg, output, step, metrics)\n                print(\"  已保存当前表现最佳的实验检查点：\", output, flush=True)\n\n    print(\"训练结束。模型权重已保存；准确率取决于实际评测，不能保证所有提问都正确。\")\n    model, vocab, cfg, data = load(output, device)\n    print(f\"采用第 {data.get('step')} 步保存的权重\")\n    for question in [\"你是谁\", \"你叫什么名字\", \"你到底是谁\", \"你是ChatGPT吗\"]:\n        print(f\"问：{question}\\n模型答：{answer(model, vocab, question, device)}\")\n\n\nif __name__ == \"__main__\":\n    p = argparse.ArgumentParser(description=\"SRT-M15 supervised QA training and evaluation\")\n    p.add_argument(\"--steps\", type=int, default=1600)\n    p.add_argument(\"--batch-size\", type=int, default=8)\n    p.add_argument(\"--context\", type=int, default=96)\n    p.add_argument(\"--dim\", type=int, default=48)\n    p.add_argument(\"--reasoning-steps\", type=int, default=2)\n    p.add_argument(\"--lr\", type=float, default=0.002)\n    p.add_argument(\"--seed\", type=int, default=42)\n    p.add_argument(\"--device\", choices=[\"auto\", \"cpu\", \"cuda\"], default=\"auto\")\n    p.add_argument(\"--log-every\", type=int, default=100)\n    p.add_argument(\"--eval-every\", type=int, default=200)\n    p.add_argument(\"--output\", type=str, default=\"checkpoints/srt_m15_qa.pt\")\n    p.add_argument(\"--eval-only\", action=\"store_true\")\n    run(p.parse_args())\n", encoding="utf-8")
Path("deepseek_web_cli.py").write_text("\"\"\"DeepSeek official WEBSITE (not API) desktop bridge.\nRun once: python deepseek_web_cli.py --login\nThen:     python deepseek_web_cli.py --ask \"介绍一下神经网络\"\n\"\"\"\nimport argparse\nfrom srt_m15.deepseek_web import DeepSeekWebClient, DeepSeekWebError\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"SRT-M15 visible DeepSeek web bridge\")\n    parser.add_argument(\"--login\", action=\"store_true\", help=\"manually sign in in visible Chromium\")\n    parser.add_argument(\"--ask\", type=str, help=\"submit one question in visible Chromium\")\n    args = parser.parse_args()\n    if not (args.login or args.ask):\n        parser.error(\"Use --login or --ask\")\n    client = DeepSeekWebClient()\n    try:\n        if args.login:\n            print(client.login())\n        if args.ask:\n            print(\"来源：DeepSeek 网页（不是 SRT-M15 本地模型）\")\n            print(client.ask(args.ask))\n    except DeepSeekWebError as exc:\n        parser.exit(2, f\"DeepSeek 网页调用失败：{exc}\\n\")\n    finally:\n        client.close()\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("tests").mkdir(exist_ok=True)
Path("tests/test_deepseek_bridge.py").write_text("\"\"\"Offline routing checks; these tests never sign in or contact DeepSeek.\"\"\"\nimport os\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nfrom srt_m15.chat import ChatReply, SRTChatEngine\nfrom srt_m15.deepseek_api import DeepSeekAPIClient, DeepSeekAPIError\nfrom srt_m15.deepseek_web import DeepSeekWebClient, DeepSeekWebError\n\n\nclass ExternalRoutingTests(unittest.TestCase):\n    def test_explicit_website_labels_source(self):\n        with tempfile.TemporaryDirectory() as root:\n            engine = SRTChatEngine(root=root, backend=\"web\")\n            fake = ChatReply(\"external\", \"由外部模型生成的回答\", \"DeepSeek Web\")\n            with patch.object(engine, \"_external_reply\", return_value=fake) as send:\n                result = engine.reply(\"什么是神经网络\")\n            self.assertEqual(result.provider, \"DeepSeek Web\")\n            send.assert_called_once_with(\"什么是神经网络\", \"web\")\n\n    def test_default_does_not_call_external_service(self):\n        with tempfile.TemporaryDirectory() as root:\n            engine = SRTChatEngine(root=root)\n            with patch.object(engine, \"_external_reply\", side_effect=AssertionError(\"called\")):\n                result = engine.reply(\"你是谁\")\n            self.assertEqual(result.provider, \"SRT-M15\")\n            self.assertEqual(result.mode, \"notice\")  # no checkpoint yet\n\n    def test_raw_forces_local_even_if_external_selected(self):\n        with tempfile.TemporaryDirectory() as root:\n            engine = SRTChatEngine(root=root, backend=\"web\")\n            with patch.object(engine, \"_external_reply\", side_effect=AssertionError(\"called\")):\n                result = engine.reply(\"/raw 你是谁\")\n            self.assertEqual(result.provider, \"SRT-M15\")\n\n    def test_auto_identity_stays_local(self):\n        with tempfile.TemporaryDirectory() as root:\n            engine = SRTChatEngine(root=root, backend=\"auto\", external_backend=\"web\")\n            with patch.object(engine, \"_external_reply\", side_effect=AssertionError(\"called\")):\n                result = engine.reply(\"你是谁\")\n            self.assertEqual(result.provider, \"SRT-M15\")\n\n    def test_auto_general_question_can_use_web(self):\n        with tempfile.TemporaryDirectory() as root:\n            engine = SRTChatEngine(root=root, backend=\"auto\", external_backend=\"web\")\n            with patch.object(engine, \"_external_reply\", return_value=ChatReply(\n                \"external\", \"回答\", \"DeepSeek Web\"\n            )) as send:\n                result = engine.reply(\"什么是神经网络\")\n            send.assert_called_once()\n            self.assertEqual(result.provider, \"DeepSeek Web\")\n\n    def test_colab_cannot_drive_logged_in_phone_browser(self):\n        with patch.dict(os.environ, {\"COLAB_RELEASE_TAG\": \"test\"}):\n            client = DeepSeekWebClient()\n            try:\n                with self.assertRaises(DeepSeekWebError):\n                    client._check_environment()\n            finally:\n                client.close()\n\n    def test_api_key_is_required_in_api_mode(self):\n        with patch.dict(os.environ, {\"DEEPSEEK_API_KEY\": \"\"}):\n            with self.assertRaises(DeepSeekAPIError):\n                DeepSeekAPIClient().ask(\"你好\")\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n", encoding="utf-8")
Path("api_server.py").write_text("\"\"\"OpenAI Chat Completions-style HTTP facade for SRT-M15.\n\nA deliberately small, single-turn, non-streaming subset of the API.\nThe \"srt-m15\" model ALWAYS calls the local trained SRT-M15 weights.\nRemote DeepSeek backends are separately named and explicitly opt-in.\n\"\"\"\nfrom contextlib import asynccontextmanager\nfrom hmac import compare_digest\nimport os\nimport threading\nimport time\nimport uuid\nfrom typing import Literal\n\nfrom fastapi import Depends, FastAPI, Header, HTTPException, status\nfrom pydantic import BaseModel, ConfigDict, Field\n\nfrom srt_m15.chat import SRTChatEngine\n\n\nLOCAL_MODEL = \"srt-m15\"\nAPI_MODEL = \"srt-m15-deepseek-api\"\nWEB_MODEL = \"srt-m15-deepseek-web\"\nTRUE_VALUES = {\"1\", \"true\", \"yes\", \"on\"}\n\n\nclass ChatMessage(BaseModel):\n    model_config = ConfigDict(extra=\"forbid\")\n    role: Literal[\"system\", \"user\", \"assistant\"]\n    content: str = Field(min_length=1, max_length=400)\n\n\nclass ChatCompletionRequest(BaseModel):\n    model_config = ConfigDict(extra=\"forbid\")\n    model: str = Field(default=LOCAL_MODEL, min_length=1)\n    messages: list[ChatMessage] = Field(min_length=1, max_length=32)\n    stream: bool = False\n    max_tokens: int = Field(default=80, ge=1, le=80)\n    temperature: float | None = Field(default=None, ge=0, le=2)\n\n\ndef create_app(*, api_key=None, engine=None, model_routes=None):\n    \"\"\"Build an app suitable for Uvicorn and for isolated TestClient tests.\n\n    Env switches SRT_API_ENABLE_DEEPSEEK_API/WEB opt external models in;\n    neither switch changes the meaning of the 'srt-m15' local model ID.\n    \"\"\"\n    configured_key = api_key if api_key is not None else os.getenv(\"SRT_API_KEY\", \"\")\n    routes = {LOCAL_MODEL: \"local\"}\n    if model_routes is not None:\n        # Explicitly injected routes are helpful for integration testing.\n        for name, backend in model_routes.items():\n            if name not in {API_MODEL, WEB_MODEL} or backend not in {\"api\", \"web\"}:\n                raise ValueError(\"Unrecognized model route\")\n            routes[name] = backend\n    else:\n        if os.getenv(\"SRT_API_ENABLE_DEEPSEEK_API\", \"\").lower() in TRUE_VALUES:\n            routes[API_MODEL] = \"api\"\n        if os.getenv(\"SRT_API_ENABLE_DEEPSEEK_WEB\", \"\").lower() in TRUE_VALUES:\n            routes[WEB_MODEL] = \"web\"\n\n    current_engine = engine if engine is not None else SRTChatEngine(\n        root=os.environ.get(\"SRT_MODEL_ROOT\") or \".\",\n        backend=\"local\",\n    )\n    inference_lock = threading.Lock()\n\n    @asynccontextmanager\n    async def lifespan(_app):\n        if not configured_key or len(configured_key) < 16:\n            raise RuntimeError(\n                \"请先配置至少 16 字符的 SRT_API_KEY；\"\n                \"不要将密钥写入 GitHub。\"\n            )\n        try:\n            yield\n        finally:\n            current_engine.close()\n\n    app = FastAPI(\n        title=\"SRT-M15 Local API\",\n        version=\"0.1.0\",\n        description=\"Small, single-turn OpenAI-compatible chat subset.\",\n        lifespan=lifespan,\n        docs_url=None,\n        redoc_url=None,\n        openapi_url=None,\n    )\n    app.state.engine = current_engine\n    app.state.model_routes = routes\n\n    def require_auth(authorization: str | None = Header(default=None)):\n        prefix = \"Bearer \"\n        if not authorization or not authorization.startswith(prefix):\n            raise HTTPException(\n                status_code=status.HTTP_401_UNAUTHORIZED,\n                detail=\"Bearer API key required\",\n                headers={\"WWW-Authenticate\": \"Bearer\"},\n            )\n        received = authorization[len(prefix):]\n        if not configured_key or not compare_digest(received, configured_key):\n            raise HTTPException(\n                status_code=status.HTTP_401_UNAUTHORIZED,\n                detail=\"Invalid API key\",\n                headers={\"WWW-Authenticate\": \"Bearer\"},\n            )\n\n    @app.get(\"/health\", dependencies=[Depends(require_auth)])\n    def health():\n        # Do not describe the service as 'trained/ready' when weights are missing.\n        metadata = current_engine.status()\n        return {\n            \"service\": \"SRT-M15\",\n            \"api\": \"running\",\n            \"local_math_checkpoint\": bool(metadata.get(\"math_checkpoint\")),\n            \"local_language_checkpoint\": bool(metadata.get(\"language_checkpoint\")),\n            \"model_ids\": list(routes),\n        }\n\n    @app.get(\"/v1/models\", dependencies=[Depends(require_auth)])\n    def list_models():\n        return {\n            \"object\": \"list\",\n            \"data\": [\n                {\"id\": name, \"object\": \"model\",\n                 \"created\": 0, \"owned_by\": \"local\" if backend == \"local\" else \"external\"}\n                for name, backend in routes.items()\n            ],\n        }\n\n    @app.post(\"/v1/chat/completions\", dependencies=[Depends(require_auth)])\n    def chat(request: ChatCompletionRequest):\n        if request.model not in routes:\n            raise HTTPException(\n                status_code=404,\n                detail=\"Unknown or disabled model; see GET /v1/models\",\n            )\n        if request.stream:\n            raise HTTPException(\n                status_code=400,\n                detail=\"Streaming is not implemented (use stream=false)\",\n            )\n        if request.temperature is not None and request.temperature != 0:\n            raise HTTPException(\n                status_code=400,\n                detail=\"Sampling temperature is not supported; omit it or use 0\",\n            )\n        if len(request.messages) != 1 or request.messages[0].role != \"user\":\n            raise HTTPException(\n                status_code=400,\n                detail=\"Only one user message is supported; multi-turn/system messages \"\n                       \"are not implemented and will not be silently discarded\",\n            )\n        question = request.messages[0].content.strip()\n        if not question:\n            raise HTTPException(status_code=400, detail=\"Question is empty\")\n        backend = routes[request.model]\n        try:\n            # Serializes calls to shared Torch weights and optional browser context.\n            with inference_lock:\n                reply = current_engine.reply(\n                    question, new_tokens=request.max_tokens, backend=backend\n                )\n        except Exception as exc:\n            # Never leak filesystem paths, auth values, or browser session details.\n            raise HTTPException(\n                status_code=503,\n                detail=f\"Inference failed ({type(exc).__name__}); check server logs\",\n            ) from exc\n\n        if reply.mode == \"error\":\n            raise HTTPException(status_code=502, detail=\"External model request failed\")\n        if reply.mode == \"notice\":\n            raise HTTPException(status_code=503, detail=reply.text)\n        if reply.mode in {\"limit\", \"vocab\"}:\n            raise HTTPException(status_code=422, detail=reply.text)\n        if not reply.text.strip():\n            raise HTTPException(status_code=503, detail=\"Model returned empty output\")\n\n        # 'srt_provider' distinguishes local learning from third-party answers.\n        return {\n            \"id\": \"chatcmpl-\" + uuid.uuid4().hex,\n            \"object\": \"chat.completion\",\n            \"created\": int(time.time()),\n            \"model\": request.model,\n            \"choices\": [{\n                \"index\": 0,\n                \"message\": {\"role\": \"assistant\", \"content\": reply.text},\n                \"finish_reason\": \"stop\",\n            }],\n            \"srt_provider\": reply.provider,\n            \"srt_mode\": reply.mode,\n        }\n\n    return app\n\n\napp = create_app()\n", encoding="utf-8")
Path("bootstrap.py").write_text("\"\"\"Install SRT-M15's bundled core source into the checked-out GitHub tree.\n\nThe mobile Colab notebook contains the original starter modules in an embedded\nZIP. This local bootstrap extracts *only* known safe source files, without\noverwriting any updated files maintained separately in this repository.\nNo network access required.\n\"\"\"\nimport base64\nimport io\nimport json\nfrom pathlib import Path\nimport re\nimport zipfile\n\nROOT = Path(__file__).resolve().parent\nNOTEBOOK = ROOT / \"SRT_M15_Phone_Colab.ipynb\"\nARCHIVE_PREFIX = \"SRT_M15_Starter/\"\nSOURCE_FILES = {\n    \"srt_m15/__init__.py\",\n    \"srt_m15/model.py\",\n    \"srt_m15/language.py\",\n    \"srt_m15/tasks.py\",\n    \"srt_m15/learning.py\",\n    \"srt_m15/exploration.py\",\n    \"srt_m15/storage.py\",\n    \"train.py\",\n    \"train_lm.py\",\n    \"generate.py\",\n    \"demo.py\",\n}\n\n\ndef bootstrap(root=ROOT):\n    root = Path(root).resolve()\n    notebook_path = root / NOTEBOOK.name\n    cells = json.loads(notebook_path.read_text(encoding=\"utf-8\"))[\"cells\"]\n    source_cell = next(\n        (\"\".join(cell.get(\"source\", [])) for cell in cells\n         if \"SOURCE_ZIP_BASE64\" in \"\".join(cell.get(\"source\", []))),\n        None,\n    )\n    if source_cell is None:\n        raise ValueError(\"Notebook 中找不到嵌入式源代码。\")\n    match = re.search(r'^SOURCE_ZIP_BASE64\\s*=\\s*\"([A-Za-z0-9+/=]+)\"',\n                      source_cell, flags=re.MULTILINE)\n    if match is None:\n        raise ValueError(\"Notebook 中未找到合法的源代码压缩包常量。\")\n    archive_bytes = base64.b64decode(match.group(1), validate=True)\n    installed, existing = [], []\n    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as package:\n        for name in sorted(SOURCE_FILES):\n            member = ARCHIVE_PREFIX + name\n            if member not in package.namelist():\n                raise ValueError(f\"源代码压缩包缺少 {member}\")\n            target = root.joinpath(*Path(name).parts)\n            if not target.resolve().is_relative_to(root):\n                raise ValueError(\"Unsafe source file path\")\n            if target.is_file():\n                existing.append(name)\n                continue\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # These are preapproved, exact source filenames; never extractall().\n            target.write_bytes(package.read(member))\n            installed.append(name)\n    return installed, existing\n\n\nif __name__ == \"__main__\":\n    new_files, skipped = bootstrap()\n    print(\"已安装缺失的底层源码:\", \", \".join(new_files) or \"无\")\n    if skipped:\n        print(\"已保留现有版本:\", \", \".join(skipped))\n    print(\"接下来可运行: pip install -r requirements_api.txt\")\n", encoding="utf-8")
Path("tests/test_api_server.py").write_text("\"\"\"Offline tests for SRT-M15's authenticated HTTP API; no model weights needed.\"\"\"\nimport os\nimport unittest\nfrom unittest.mock import patch\n\nfrom fastapi.testclient import TestClient\n\n# Importing the module does NOT start the server or contact any external provider.\nfrom api_server import (\n    API_MODEL, LOCAL_MODEL, WEB_MODEL, create_app,\n)\nfrom srt_m15.chat import ChatReply\n\n\nKEY = \"test-local-secret-1234567890\"\n\n\nclass FakeEngine:\n    def __init__(self):\n        self.calls = []\n        self.closed = False\n        self.next = ChatReply(\"text\", \"我是SRT-M15。\", \"SRT-M15\")\n\n    def reply(self, message, *, new_tokens=80, backend=None):\n        self.calls.append((message, new_tokens, backend))\n        return self.next\n\n    def status(self):\n        return {\n            \"math_checkpoint\": True,\n            \"language_checkpoint\": True,\n        }\n\n    def close(self):\n        self.closed = True\n\n\nclass APITests(unittest.TestCase):\n    def setUp(self):\n        self.engine = FakeEngine()\n        # Explicit route opt-in is intentionally NOT supplied.\n        self.app = create_app(api_key=KEY, engine=self.engine, model_routes={})\n        self.client_ctx = TestClient(self.app)\n        self.client = self.client_ctx.__enter__()\n        self.auth = {\"Authorization\": f\"Bearer {KEY}\"}\n\n    def tearDown(self):\n        self.client_ctx.__exit__(None, None, None)\n\n    def test_rejects_missing_and_bad_keys(self):\n        for headers in ({}, {\"Authorization\": \"Bearer wrong\"}):\n            result = self.client.get(\"/v1/models\", headers=headers)\n            self.assertEqual(result.status_code, 401)\n        self.assertEqual(self.engine.calls, [])\n\n    def test_lists_only_local_model_by_default(self):\n        resp = self.client.get(\"/v1/models\", headers=self.auth)\n        self.assertEqual(resp.status_code, 200)\n        self.assertEqual([v[\"id\"] for v in resp.json()[\"data\"]], [LOCAL_MODEL])\n\n    def test_chat_shape_and_provider(self):\n        resp = self.client.post(\n            \"/v1/chat/completions\", headers=self.auth,\n            json={\"model\": LOCAL_MODEL,\n                  \"messages\": [{\"role\": \"user\", \"content\": \"你是谁\"}],\n                  \"max_tokens\": 30},\n        )\n        self.assertEqual(resp.status_code, 200, resp.text)\n        data = resp.json()\n        self.assertEqual(data[\"object\"], \"chat.completion\")\n        self.assertEqual(data[\"choices\"][0][\"message\"][\"content\"], \"我是SRT-M15。\")\n        self.assertEqual(data[\"srt_provider\"], \"SRT-M15\")\n        self.assertEqual(self.engine.calls, [(\"你是谁\", 30, \"local\")])\n\n    def test_requires_single_turn_instead_of_dropping_history(self):\n        body = {\"model\": LOCAL_MODEL, \"messages\": [\n            {\"role\": \"system\", \"content\": \"请简短回答\"},\n            {\"role\": \"user\", \"content\": \"你是谁\"},\n        ]}\n        resp = self.client.post(\"/v1/chat/completions\", headers=self.auth, json=body)\n        self.assertEqual(resp.status_code, 400)\n        self.assertIn(\"multi-turn\", resp.json()[\"detail\"])\n        self.assertEqual(self.engine.calls, [])\n\n    def test_rejects_streaming_and_unsupported_temperature(self):\n        base = {\"model\": LOCAL_MODEL,\n                \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}]}\n        for extra in ({\"stream\": True}, {\"temperature\": 0.7}):\n            res = self.client.post(\"/v1/chat/completions\",\n                                   headers=self.auth, json={**base, **extra})\n            self.assertEqual(res.status_code, 400)\n        self.assertEqual(self.engine.calls, [])\n\n    def test_external_model_is_not_exposed_by_default(self):\n        resp = self.client.post(\"/v1/chat/completions\", headers=self.auth, json={\n            \"model\": API_MODEL,\n            \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}],\n        })\n        self.assertEqual(resp.status_code, 404)\n\n    def test_disabled_model_not_routed_even_if_requested(self):\n        resp = self.client.post(\"/v1/chat/completions\", headers=self.auth, json={\n            \"model\": WEB_MODEL,\n            \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}],\n        })\n        self.assertEqual(resp.status_code, 404)\n\n    def test_unavailable_weights_return_service_unavailable(self):\n        self.engine.next = ChatReply(\"notice\", \"请训练并加载权重\")\n        res = self.client.post(\"/v1/chat/completions\", headers=self.auth, json={\n            \"model\": LOCAL_MODEL, \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}],\n        })\n        self.assertEqual(res.status_code, 503)\n\n    def test_limited_input_return_422(self):\n        self.engine.next = ChatReply(\"vocab\", \"词表缺少字符\")\n        res = self.client.post(\"/v1/chat/completions\", headers=self.auth, json={\n            \"model\": LOCAL_MODEL, \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}],\n        })\n        self.assertEqual(res.status_code, 422)\n\n    def test_health_requires_auth(self):\n        self.assertEqual(self.client.get(\"/health\").status_code, 401)\n        res = self.client.get(\"/health\", headers=self.auth)\n        self.assertEqual(res.status_code, 200)\n        self.assertTrue(res.json()[\"local_language_checkpoint\"])\n\n    def test_explicitly_enabled_external_model_has_distinct_source(self):\n        fake = FakeEngine()\n        fake.next = ChatReply(\"external\", \"远程回答\", \"DeepSeek API\")\n        app = create_app(api_key=KEY, engine=fake,\n                         model_routes={API_MODEL: \"api\"})\n        with TestClient(app) as client:\n            models = client.get(\"/v1/models\", headers=self.auth).json()[\"data\"]\n            self.assertEqual(set(x[\"id\"] for x in models),\n                             {LOCAL_MODEL, API_MODEL})\n            res = client.post(\"/v1/chat/completions\", headers=self.auth, json={\n                \"model\": API_MODEL, \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}],\n            })\n            self.assertEqual(res.status_code, 200)\n            self.assertEqual(res.json()[\"srt_provider\"], \"DeepSeek API\")\n            self.assertEqual(fake.calls, [(\"你好\", 80, \"api\")])\n\n    def test_external_failure_returns_bad_gateway(self):\n        fake = FakeEngine()\n        fake.next = ChatReply(\"error\", \"远端超时\", \"DeepSeek API\")\n        app = create_app(api_key=KEY, engine=fake,\n                         model_routes={API_MODEL: \"api\"})\n        with TestClient(app) as client:\n            res = client.post(\"/v1/chat/completions\", headers=self.auth, json={\n                \"model\": API_MODEL, \"messages\": [{\"role\": \"user\", \"content\": \"你好\"}],\n            })\n            self.assertEqual(res.status_code, 502)\n            self.assertNotIn(\"远端超时\", res.text)\n\n    def test_api_key_must_be_configured(self):\n        app = create_app(api_key=\"\", engine=FakeEngine())\n        with self.assertRaises(RuntimeError):\n            with TestClient(app):\n                pass\n\n    def test_shutdown_releases_engine(self):\n        # The engine shared by the app created in setUp is closed in tearDown.\n        app = create_app(api_key=KEY, engine=FakeEngine())\n        with TestClient(app):\n            self.assertFalse(app.state.engine.closed)\n        self.assertTrue(app.state.engine.closed)\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n", encoding="utf-8")
print("已安装：SRT-M15 本地模型、多来源聊天和 HTTP API（FastAPI）模块。")


## 第 7 步（可选）：从已保存的 ZIP 恢复模型权重

如果你在新的 Colab 会话里没有重新训练，请运行此单元并上传之前保存的 `SRT_M15_Results.zip`。**如果当前会话已经训练过，跳过此单元。**

In [ ]:
# 可选：使用手机已有的 SRT_M15_Results.zip 恢复上次训练的权重，不必重训
# 如果前面的训练步骤已经完成，跳过此单元即可。
import io
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

uploaded = files.upload()
safe_names = {"srt_m15_toy.pt", "srt_m15_lm.pt", "srt_m15_qa.pt"}
Path("checkpoints").mkdir(exist_ok=True)
for name, binary in uploaded.items():
    if not name.lower().endswith(".zip"):
        print("略过非 ZIP 文件:", name)
        continue
    with ZipFile(io.BytesIO(binary)) as z:
        for entry in z.namelist():
            basename = Path(entry).name
            if entry in (f"checkpoints/{x}" for x in safe_names) and not entry.endswith("/"):
                (Path("checkpoints") / basename).write_bytes(z.read(entry))
                print("已恢复:", basename)


## 第 8 步：打开手机聊天窗口

可以选择 **SRT-M15 本地模型**、**DeepSeek 网页（需要本地电脑）**、**DeepSeek API** 或 **自动路由**。

默认只使用 SRT-M15，不向第三方发送提问。切换到外部模式后，界面会显示答案的真实来源。想验证 SRT-M15 是否自己学会了“你是谁”，请先完成第 9 步训练，然后重新执行本步骤；外部模型回答不能当作本地模型能力。


In [ ]:
%run chat_ui.py


In [ ]:
from srt_m15.chat import SRTChatEngine
engine = SRTChatEngine(backend="local")
for question in ("你是谁", "1+9=多少"):
    answer = engine.reply(question)
    print(f"提问：{question}\n来源：{answer.provider}\n回答：{answer.text}\n")


## 第 9 步：用监督学习训练模型自己回答身份问题

通过 `train_qa.py` 的“问题作为上下文、只监督答案字符”方式训练 SRT-M15。脚本包含多种身份问题训练样例和未参与训练的验证问法，并输出真实神经网络生成结果。

与之前的 260 步字符续写不同，新训练使用专门的答案损失和确定性解码。训练结束会将效果较好的权重保存到 `checkpoints/srt_m15_qa.pt`。**小模型不保证能准确泛化，需要查看输出日志。**


In [ ]:
# 检查代码语法与离线安全路由，不触碰 DeepSeek 账号或密钥
!python -m py_compile srt_m15/chat.py srt_m15/deepseek_web.py srt_m15/deepseek_api.py train_qa.py chat_ui.py deepseek_web_cli.py api_server.py bootstrap.py
!python -m unittest discover -s tests -p "test_deepseek_bridge.py" -v
print("语法及离线路由检查结束；网页版真实调用还需要本机浏览器登录。")


In [ ]:
# 监督答案训练（手机 Colab 可运行，不使用 DeepSeek API）
!python train_qa.py --steps 1600 --context 96 --batch-size 8 --device auto --output checkpoints/srt_m15_qa.pt --log-every 200 --eval-every 200


**训练完成后请重新执行第 8 步的 `%run chat_ui.py`**，新建聊天引擎会优先加载 `checkpoints/srt_m15_qa.pt`。输入“你是谁”和其他问法观察真实生成；输入 `/raw 你是谁` 也可以强制只用本地模型。如果你使用了旧权重，必须重新训练后才能应用新训练方式。


## 第 10 步：下载模型权重和训练记录

训练完成后，下载检查点保存到手机/云盘。Colab 会话中的文件可能被清理；下载不包括登录凭证。


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
output = Path("/content/SRT_M15_Results.zip")
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["srt_m15_toy.pt", "srt_m15_lm.pt", "srt_m15_qa.pt", "metrics.json", "knowledge.json"]:
        path = Path("checkpoints") / name
        if path.exists():
            z.write(path, arcname=path.as_posix())
print("已收集：", output)
files.download(str(output))


## 第 11 步（可选）：DeepSeek 外部模型接入说明

* **无需 API Key 的网页版：** 在本机电脑使用 `pip install playwright`、`python -m playwright install chromium`，然后运行 `python deepseek_web_cli.py --login`，自己登录 DeepSeek，使用 `--ask` 提问。**Google Colab 无法操控你手机浏览器，网页版选项在云端不可用。**
* **可在手机 Colab 使用的 API：** 如果拥有 DeepSeek API Key，可以执行下一单元安全输入，然后在第 8 步选择“DeepSeek API”。
* 外部回答不会自动改变 SRT-M15 的模型权重。


In [ ]:
# 只有你选择 DeepSeek 官方 API 时才运行（可选），请勿将密钥上传 GitHub。
from getpass import getpass
import os
os.environ["DEEPSEEK_API_KEY"] = getpass("DeepSeek API Key（输入不回显）：")
print("API Key 仅写入当前 Colab 运行进程的环境变量。")


## 第 12 步（可选）：测试 SRT-M15 的 OpenAI 风格本地 API

本步骤先安装 FastAPI/Uvicorn/httpx，再在当前 Colab 进程内模拟 HTTP 请求验证接口，而不是创建公网服务。运行前请先执行第 6 步安装代码；实际文本问答仍需要第 9 步训练或恢复语言模型权重。

手机无法直接用 `127.0.0.1:8000` 连接 Colab 云服务器；想让其他应用远程调用，需在自己持续运行的电脑/服务器上安全部署。请勿把 API Key 提交到 GitHub。


In [ ]:
%pip -q install 'fastapi>=0.115,<1' 'pydantic>=2.8,<3' 'uvicorn>=0.30,<1' 'httpx>=0.27,<1'
!python -m unittest discover -s tests -p 'test_api_server.py' -v


In [ ]:
import os
import secrets
from fastapi.testclient import TestClient
from api_server import create_app
# 临时内存密钥，不写入代码或 GitHub
temporary_api_key = secrets.token_urlsafe(32)
with TestClient(create_app(api_key=temporary_api_key)) as client:
    headers = {'Authorization': 'Bearer ' + temporary_api_key}
    print('模型列表:', client.get('/v1/models', headers=headers).json())
    print('运行状态:', client.get('/health', headers=headers).json())
    # 当前存在 checkpoints/srt_m15_qa.pt 才会得到神经网络文本回答
    result = client.post('/v1/chat/completions', headers=headers,
                         json={'model': 'srt-m15',
                               'messages': [{'role': 'user', 'content': '你是谁'}],
                               'stream': False})
    print('HTTP状态:', result.status_code, '响应:', result.json())
